In [10]:
import pandas as pd
from sklearn.neighbors import kneighbors_graph
import networkx as nx

In [2]:
norbit=pd.read_csv("/mnt/isilon/tan_lab/yanga11/CytoIntermediates/HyperGlioma/Nolan/pHGG_Nolan.csv")

In [3]:
norbit.head()

,Sample_Name,x.coord,y.coord,Cell_Type,TCN_Label
0,Nolan_7316-161,20940.941783,17216.245394,Mesenchymal-1 Tumor Cells,8
1,Nolan_7316-161,20949.031457,17235.697020,Mesenchymal-1 Tumor Cells,8
2,Nolan_7316-161,20781.909836,17239.840164,Mesenchymal-1 Tumor Cells,8
3,Nolan_7316-161,21004.918956,17244.803571,Mesenchymal-1 Tumor Cells,8
4,Nolan_7316-161,21027.561551,17240.605396,Mesenchymal-1 Tumor Cells,8


In [6]:
compartments=norbit.TCN_Label.unique()
compartments

array([ 8, 13,  5, 11,  4,  9,  6,  2,  3, 15, 12, 10,  1, 14,  7])

In [11]:
cells={"Sample_Name": [], "x.coord": [], "y.coord": [], "Cell_Type": [], "TCN_Label": [], "Instance": []}

for s in sorted(set(norbit["Sample_Name"])):
    sample=norbit[norbit["Sample_Name"]==s]
    print(s)
    A=kneighbors_graph(sample[["x.coord","y.coord"]], 25, mode='connectivity', include_self=False)
    G=nx.from_numpy_array(A)
    nx.set_node_attributes(G, dict(enumerate(sample['x.coord'])),'x')
    nx.set_node_attributes(G, dict(enumerate(sample['y.coord'])),'y')
    nx.set_node_attributes(G, dict(enumerate(sample['Cell_Type'])),'Cell Type')
    nx.set_node_attributes(G, dict(enumerate(sample['TCN_Label'])),'TCN_Label')
    for compartment in compartments:
        G_c=G.subgraph([x for x,y in G.nodes(data=True) if y['TCN_Label']==compartment])
        instances=nx.connected_components(G_c)
        index=0
        for instance in instances:
            if len(instance) >= 40: #this value filters out instances with less than 40 same-instance neighbors to increase neighborhood continuity
                G_ci=G.subgraph(instance)
                for node in G_ci.nodes:
                    cells["Sample_Name"].append(s)
                    cells['x.coord'].append(G_ci.nodes[node]['x'])
                    cells['y.coord'].append(G_ci.nodes[node]['y'])
                    cells['Cell_Type'].append(G_ci.nodes[node]['Cell Type'])
                    cells['TCN_Label'].append(G_ci.nodes[node]['TCN_Label'])
                    cells['Instance'].append(str(G_ci.nodes[node]['TCN_Label']) + '-' + str(index))
                index +=1
cells=pd.DataFrame(cells)


Nolan_7316-161


KeyboardInterrupt: 

In [9]:
cells.to_csv("/mnt/isilon/tan_lab/yanga11/CytoIntermediates/HyperGlioma/Nolan/Nolan/Instances/NolanInstances.csv")